In [17]:
import pandas as pd

df = pd.read_csv("train.csv")

In [18]:
features = [
    "OverallQual",
    "GrLivArea",
    "TotalBsmtSF",
    "GarageCars",
    "YearBuilt",
    "Neighborhood"
]

target = "SalePrice"

df = df[features + [target]]


In [19]:
df["TotalBsmtSF"].fillna(0, inplace=True)
df["GarageCars"].fillna(0, inplace=True)
df["Neighborhood"].fillna("Unknown", inplace=True)


/tmp/ipython-input-2253027999.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalBsmtSF"].fillna(0, inplace=True)
/tmp/ipython-input-2253027999.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df

In [20]:
from sklearn.model_selection import train_test_split

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

numeric_features = [
    "OverallQual", "GrLivArea", "TotalBsmtSF",
    "GarageCars", "YearBuilt"
]

categorical_features = ["Neighborhood"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])


In [22]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['OverallQual', 'GrLivArea',
                                                   'TotalBsmtSF', 'GarageCars',
                                                   'YearBuilt']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Neighborhood'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")


MAE: 18417.43
MSE: 818470597.03
RMSE: 28608.93
R²: 0.893


In [24]:
import joblib
from pathlib import Path

model_path = Path("house_price_model_1.pkl")
joblib.dump(pipeline, model_path)


['house_price_model_1.pkl']

In [25]:
from google.colab import files

files.download('house_price_model_1.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>